# K-Nearest Neighbors (KNN) from Scratch

**Dataset:** Iris (sklearn built-in)  
**Task:** Multiclass classification — 3 species of iris flowers

---

## Overview

**K-Nearest Neighbors** is a **non-parametric, lazy learning** algorithm — it stores all training data and makes predictions at query time rather than learning an explicit model. There is no training phase beyond storing the data.

### Prediction

To classify a new point $\mathbf{x}$:
1. Compute the distance from $\mathbf{x}$ to every training point
2. Select the $K$ closest training points (the **K-nearest neighbors**)
3. Return the **majority class** among those $K$ neighbors

### Distance Metric

The most common choice is **Euclidean distance**:

$$d(\mathbf{x}, \mathbf{x}') = \sqrt{\sum_{j=1}^p (x_j - x_j')^2} = \|\mathbf{x} - \mathbf{x}'\|_2$$

Other options include Manhattan ($L_1$) distance and Minkowski distance (generalization of both).

### Choosing K

- **Small K** (e.g., K=1): very flexible, fits training data perfectly — prone to overfitting and noise
- **Large K**: smoother decision boundaries — higher bias, lower variance
- **Rule of thumb**: start with $K = \sqrt{n}$ and tune via cross-validation

### Complexity

- Training: $O(1)$ (just store the data)
- Prediction: $O(n \cdot p)$ per query — can be slow for large datasets

## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 6)
plt.rcParams['font.size'] = 12

## 2. Load and Explore the Dataset

In [ ]:
data = load_iris()
X, y = data.data, data.target
feature_names = data.feature_names
target_names  = data.target_names

print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Classes: {target_names}")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
df = pd.DataFrame(X, columns=feature_names)
df['species'] = [target_names[i] for i in y]
df.groupby('species').agg(['mean', 'std']).round(2)

In [ ]:
colors = ['#e74c3c', '#2ecc71', '#3498db']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for cls, color, label in zip([0, 1, 2], colors, target_names):
    mask = y == cls
    axes[0].scatter(X[mask, 0], X[mask, 1], c=color, label=label, alpha=0.8, edgecolors='k', s=60)
    axes[1].scatter(X[mask, 2], X[mask, 3], c=color, label=label, alpha=0.8, edgecolors='k', s=60)

axes[0].set_xlabel(feature_names[0]); axes[0].set_ylabel(feature_names[1])
axes[0].set_title('Sepal Features'); axes[0].legend()
axes[1].set_xlabel(feature_names[2]); axes[1].set_ylabel(feature_names[3])
axes[1].set_title('Petal Features'); axes[1].legend()

plt.suptitle('Iris Dataset Exploration', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. KNN from Scratch

We implement KNN using only NumPy — no scikit-learn models.

In [ ]:
class KNNScratch:
    """K-Nearest Neighbors classifier using Euclidean distance."""

    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self

    def predict(self, X):
        X = np.array(X)
        predictions = []
        for x in X:
            # Euclidean distances to all training points
            distances = np.linalg.norm(self.X_train - x, axis=1)
            # Indices of K nearest neighbors
            k_nearest = np.argsort(distances)[:self.k]
            # Majority vote
            k_labels = self.y_train[k_nearest]
            vote = np.bincount(k_labels).argmax()
            predictions.append(vote)
        return np.array(predictions)

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == np.array(y))

## 4. Train/Test Split and Scaling

KNN is a distance-based algorithm — features with larger scales dominate. **StandardScaler** normalizes each feature to zero mean and unit variance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape[0]} | Test: {X_test_s.shape[0]}")

## 5. Train and Evaluate

In [ ]:
knn = KNNScratch(k=3)
knn.fit(X_train_s, y_train)

train_acc = knn.accuracy(X_train_s, y_train)
test_acc  = knn.accuracy(X_test_s,  y_test)

print(f"K = 3")
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test  accuracy: {test_acc:.4f}")

## 6. Confusion Matrix

In [ ]:
y_pred = knn.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)
ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
ax.set_xticklabels(target_names, rotation=20)
ax.set_yticklabels(target_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (K=3)')

for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=13)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=target_names))

## 7. Effect of K on Accuracy

We sweep K from 1 to 30 and record train/test accuracy. K=1 always achieves 100% training accuracy (a point's nearest neighbor is itself), but generalizes poorly.

In [ ]:
k_range = range(1, 31)
train_accs, test_accs = [], []

for k in k_range:
    model = KNNScratch(k=k)
    model.fit(X_train_s, y_train)
    train_accs.append(model.accuracy(X_train_s, y_train))
    test_accs.append(model.accuracy(X_test_s,   y_test))

best_k = k_range.start + np.argmax(test_accs)

plt.figure(figsize=(10, 5))
plt.plot(k_range, train_accs, 'o-', color='steelblue', label='Train accuracy', lw=2)
plt.plot(k_range, test_accs,  's--', color='tomato',    label='Test accuracy',  lw=2)
plt.axvline(best_k, color='gray', linestyle=':', label=f'Best K={best_k}')
plt.xlabel('K (number of neighbors)')
plt.ylabel('Accuracy')
plt.title('KNN: Effect of K on Train and Test Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best test accuracy: {max(test_accs):.4f} at K={best_k}")

## 8. Decision Boundary Visualization (2D)

We use only the two petal features to visualize how KNN partitions feature space. Notice how the boundary becomes smoother as K increases.

In [ ]:
# Use petal features for 2D visualization
X2 = X[:, 2:4]
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y
)
scaler2 = StandardScaler()
X2_train_s = scaler2.fit_transform(X2_train)
X2_test_s  = scaler2.transform(X2_test)

x_min, x_max = X2_train_s[:, 0].min() - 0.5, X2_train_s[:, 0].max() + 0.5
y_min, y_max = X2_train_s[:, 1].min() - 0.5, X2_train_s[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

k_vals = [1, 5, 15]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, k in zip(axes, k_vals):
    model = KNNScratch(k=k)
    model.fit(X2_train_s, y2_train)
    Z = model.predict(grid).reshape(xx.shape)
    acc = model.accuracy(X2_test_s, y2_test)

    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlBu')
    for cls, color, label in zip([0, 1, 2], colors, target_names):
        mask = y2_train == cls
        ax.scatter(X2_train_s[mask, 0], X2_train_s[mask, 1],
                   c=color, label=label, edgecolors='k', s=50, alpha=0.9)
    ax.set_title(f'K={k}  (test acc={acc:.2f})')
    ax.set_xlabel(feature_names[2])
    if ax is axes[0]:
        ax.set_ylabel(feature_names[3])
        ax.legend(fontsize=8)

plt.suptitle('KNN Decision Boundaries for Different K', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Effect of Feature Scaling

KNN is highly sensitive to feature scale. Unscaled features can cause one feature to dominate all distance calculations.

In [ ]:
results = {}
for k in [1, 3, 5, 10, 15]:
    # Unscaled
    m_unscaled = KNNScratch(k=k).fit(X_train, y_train)
    acc_unscaled = m_unscaled.accuracy(X_test, y_test)

    # Scaled
    m_scaled = KNNScratch(k=k).fit(X_train_s, y_train)
    acc_scaled = m_scaled.accuracy(X_test_s, y_test)

    results[k] = (acc_unscaled, acc_scaled)
    print(f"K={k:2d} | Unscaled: {acc_unscaled:.4f} | Scaled: {acc_scaled:.4f}")

k_list = list(results.keys())
unscaled_accs = [results[k][0] for k in k_list]
scaled_accs   = [results[k][1] for k in k_list]

plt.figure(figsize=(8, 5))
plt.plot(k_list, unscaled_accs, 'o--', color='tomato',    label='Unscaled features')
plt.plot(k_list, scaled_accs,   's-',  color='steelblue', label='StandardScaler')
plt.xlabel('K'); plt.ylabel('Test Accuracy')
plt.title('KNN: Impact of Feature Scaling')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Using `mlpackage`

The custom `mlpackage` includes a KNN implementation with identical logic plus built-in visualization helpers.

In [ ]:
import sys
sys.path.insert(0, '../../../')

from mlpackage.supervised_learning.knn import KNN

pkg_knn = KNN(k=3)
pkg_knn.fit(X_train_s, y_train)

print(f"mlpackage KNN accuracy: {pkg_knn.accuracy(X_test_s, y_test):.4f}")
print("\nConfusion Matrix:")
print(pkg_knn.confusion_matrix(X_test_s, y_test))

## 11. Summary

### Key Takeaways

- KNN is a **lazy learner** — training is trivial (just store data), but prediction requires comparing to every training point.
- The **Euclidean distance** is the standard metric, but KNN works with any valid distance function.
- **K=1** perfectly memorizes training data (overfitting); larger K gives smoother, more regularized boundaries.
- **Feature scaling is essential** — without it, features with large ranges dominate distance calculations.
- The **decision boundary** is non-linear and adapts to local structure, unlike linear classifiers.
- KNN works well for small datasets but becomes expensive at prediction time for large $n$.